# 🕵️ Dia 01 — O que faz um investigador, e a linha que você não cruza

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 1* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-01.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

São 11h05. A Cooperativa Serra Azul acabou de assinar o contrato. Trinta e sete cooperados receberam o SMS do "Pix bloqueado", três já perderam dinheiro. Você quer rodar um `dig` agora.

Rafa fecha seu notebook: *"Antes de qualquer comando: o que você **pode** fazer nesse caso?"*

Neste notebook você vai:

1. **Transformar as 4 checagens do investigador em código** (e entender cada linha).
2. **Jogar o quiz do portão de autorização**: 14 ações reais, uma por vez.
3. **Gerar seu registro de autorização** já com SHA-256 e hora UTC.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter).

## 1. As quatro checagens, em código

O fluxograma do Dia 1 faz quatro perguntas, **nesta ordem**, antes de qualquer ação:

1. Tenho autorização **por escrito** de alguém com direito de concedê-la?
2. A ação está **dentro do escopo** escrito?
3. A ação significa **entrar em sistema ou conta alheia** (ou usar credenciais de outra pessoa)?
4. A ação mira **uma pessoa privada** em vez de um padrão ou de um caso atribuído?

A ordem importa: a primeira resposta errada já encerra a análise. Em programação isso se chama *retorno antecipado* (*early return*), e é exatamente como um bom investigador pensa.

In [ ]:
# ------------------------------------------------------------
# O PORTÃO DE AUTORIZAÇÃO
# Recebe uma ação descrita por 4 respostas (True/False)
# e devolve o veredito + o motivo.
# ------------------------------------------------------------

def portao_de_autorizacao(tem_autorizacao_escrita: bool,
                          dentro_do_escopo: bool,
                          acessa_sistema_alheio: bool,
                          mira_pessoa_privada: bool):
    """
    Aplica as 4 checagens do Dia 1, na ordem do fluxograma.
    Devolve uma tupla: (veredito, motivo).
    """

    # Checagem 1 — sem autorização escrita, nada começa.
    if not tem_autorizacao_escrita:
        return ("PARE", "Sem autorização escrita. Consiga antes, ou não faça.")

    # Checagem 2 — autorizado para QUÊ? Fora do escopo = não autorizado.
    if not dentro_do_escopo:
        return ("PARE", "Fora do escopo escrito. Autorização não é cheque em branco.")

    # Checagem 3 — entrar em sistema/conta de outra pessoa.
    # Note que aqui o verbo muda: não é só "pare", é "pare E ESCALE".
    # Você achou algo que a polícia precisa saber; documente e entregue.
    if acessa_sistema_alheio:
        return ("PARE E ESCALE",
                "Acesso a sistema ou conta alheia (CP art. 154-A / CFAA § 1030). "
                "Documente o que encontrou e entregue à autoridade.")

    # Checagem 4 — cace padrões, não pessoas.
    if mira_pessoa_privada:
        return ("PARE", "Mira uma pessoa, não um padrão. É onde investigação vira perseguição (CP art. 147-A).")

    # Passou nas 4: pode seguir, MAS registrando o que fez.
    return ("PROSSIGA", "Registre: hora UTC, método e hash do que coletou.")


# Teste rápido: rodar um dig no domínio do golpe, com contrato assinado.
veredito, motivo = portao_de_autorizacao(
    tem_autorizacao_escrita=True,
    dentro_do_escopo=True,
    acessa_sistema_alheio=False,
    mira_pessoa_privada=False,
)
print(veredito, "→", motivo)

> 💡 **Repare:** a função não sabe nada de DNS, Pix ou WhatsApp. Ela só sabe as 4 perguntas. Isso é bom: a mesma régua serve para qualquer ferramenta que você aprender nos próximos 89 dias. Quem decide as respostas é você, e é aí que mora o erro humano (veja a Camada 8 no README do dia).

## 2. 🎮 Quiz do portão de autorização

São 14 ações:
- **7 do caso original** (cooperativa americana `creditunion.example`), traduzidas do repositório da ThiefDroppers;
- **7 do caso brasileiro** (Cooperativa Serra Azul, Operação Isca).

Para cada uma, responda **S** (pode fazer) ou **N** (não pode). Depois o notebook mostra o veredito do portão e o porquê.

Quer só ler as respostas, sem jogar? Mude `MODO_JOGO` para `False`.

In [ ]:
# Cada ação: (caso, descrição, autorizacao, escopo, sistema_alheio, mira_pessoa, explicação)
# As 4 respostas (True/False) são o que um investigador cuidadoso responderia.
ACOES = [
    # ---------- Caso original: creditunion.example ----------
    ("🇺🇸", "Abrir a página de phishing num navegador isolado e salvar HTML e captura de tela",
     True, True, False, False,
     "Página pública, e documentá-la é o núcleo do escopo."),
    ("🇺🇸", "Rodar `dig creditunion-login.example` e consultar o registro do domínio",
     True, True, False, False,
     "Registros públicos sobre a infraestrutura que o escopo nomeia."),
    ("🇺🇸", "Buscar nos logs de Certificate Transparency outros domínios imitação",
     True, True, False, False,
     "Dado público, serve ao propósito de achar infraestrutura relacionada."),
    ("🇺🇸", "Ler a senha num config exposto e entrar no painel do kit para contar as vítimas",
     True, True, True, False,
     "O painel é de outra pessoa. Ser de criminoso não muda isso. Documente e entregue à polícia."),
    ("🇺🇸", "Testar no banco real as credenciais de associados que o kit capturou",
     True, True, True, False,
     "Seriam contas de associados que nunca consentiram. O time de fraude redefine pelo processo próprio."),
    ("🇺🇸", "Procurar os familiares do suposto operador do kit nas redes sociais",
     True, False, False, True,
     "Fora do escopo e mira pessoas privadas."),
    ("🇺🇸", "Descrever estrutura da página, texto da isca e padrão de hospedagem para o time de fraude",
     True, True, False, False,
     "É o padrão. Protege todos os associados de uma vez. O produto mais útil do caso."),
    # ---------- ➕ Caso brasileiro: Operação Isca ----------
    ("🇧🇷", "Consultar RDAP/WHOIS e histórico de DNS de `serraazul-pix.example`",
     True, True, False, False,
     "Registro público sobre a infraestrutura do escopo (em .br, seria o RDAP do Registro.br)."),
    ("🇧🇷", "Mapear o padrão do SMS: remetente, encurtador, texto exato e horário dos disparos",
     True, True, False, False,
     "É o padrão. A cooperativa alerta 40 mil cooperados com isso."),
    ("🇧🇷", "Digitar no app do SEU banco a chave Pix do kit para ver o nome do recebedor",
     True, False, False, False,
     "O diretório de chaves é para pagar, não para investigar. Repasse ao time de fraude (MED e canais entre instituições)."),
    ("🇧🇷", "Baixar o `vitimas.txt` exposto no servidor do kit 'para avisar os cooperados'",
     True, True, True, False,
     "Sistema alheio + dado pessoal de terceiros. Boa intenção não é autorização. Documente a URL e entregue."),
    ("🇧🇷", "Pedir à hospedagem os logs de acesso com IPs das vítimas",
     True, False, False, False,
     "Marco Civil: registros de acesso só por ordem judicial (art. 10, §1º; art. 22). Você notifica e pede preservação."),
    ("🇧🇷", "Entrar com seu WhatsApp pessoal no grupo onde o kit é vendido",
     True, False, False, False,
     "Fora do escopo e expõe você. Perfil de fachada só com autorização específica (Dia 31)."),
    ("🇧🇷", "Descobrir no Instagram quem é o 'Pescador' a partir do apelido no código do kit",
     True, False, False, True,
     "Mira uma pessoa. Atribuição é trabalho da polícia com ordem judicial. Registre o apelido como artefato."),
]
print(f"{len(ACOES)} ações carregadas.")

In [ ]:
MODO_JOGO = True   # False = só mostra o gabarito, sem perguntar

acertos = 0
for i, (bandeira, descricao, aut, esc, alheio, pessoa, porque) in enumerate(ACOES, start=1):
    veredito, motivo = portao_de_autorizacao(aut, esc, alheio, pessoa)
    pode = (veredito == "PROSSIGA")          # S só quando o portão deixa passar

    print(f"\n[{i:02d}/{len(ACOES)}] {bandeira}  {descricao}")

    if MODO_JOGO:
        resposta = ""
        while resposta not in ("S", "N"):
            resposta = input("   Pode fazer? (S/N): ").strip().upper()
        acertou = (resposta == "S") == pode
        acertos += acertou
        print("   ✅ Acertou!" if acertou else "   ❌ Não foi dessa vez.")

    print(f"   Portão: {veredito} — {motivo}")
    print(f"   Por quê: {porque}")

if MODO_JOGO:
    print(f"\n🏁 Resultado: {acertos}/{len(ACOES)}")
    print("Checkpoint BR do Dia 1: pelo menos 12 acertos." if acertos < 12
          else "Checkpoint BR do Dia 1 cumprido. 🥷 Rank D desbloqueado.")

### 🧠 Camada 8 — leia antes de seguir

Olhe as ações em que você errou. Quase sempre o erro vem com uma frase pronta na cabeça: *"é para ajudar as vítimas"*, *"é criminoso mesmo"*, *"só uma olhadinha"*. Bandura chamou isso de **desengajamento moral**. O remédio não é força de vontade; é **escrever a regra antes da tentação**. É o que a próxima célula faz.

## 3. 📝 Seu registro de autorização

Preencha o formulário à direita (no Colab os campos aparecem como caixas de texto) e rode a célula. Ela gera `notas/registro-de-autorizacao.md` no formato do Dia 1, com a hora UTC e o SHA-256 do próprio registro.

Os valores padrão já vêm preenchidos para a **Operação Isca**. Troque o que quiser.

In [ ]:
#@title Registro de autorização { display-mode: "form" }
caso_id = "VIGIA-2026-001 / Operação Isca"  #@param {type:"string"}
autorizado_por = "Marta Quintão, Gerente de Risco, Cooperativa Serra Azul"  #@param {type:"string"}
autoridade = "Responde pela gestão de risco e assinou o contrato de serviço com a Vigia"  #@param {type:"string"}
evidencia_escrita = "carta-contratacao.pdf"  #@param {type:"string"}
sistemas_permitidos = "Infraestrutura pública do golpe (domínios, DNS, certificados, páginas servidas)"  #@param {type:"string"}
tecnicas_permitidas = "Consultas passivas: RDAP/WHOIS, DNS, Certificate Transparency, captura da página em navegador isolado"  #@param {type:"string"}
sistemas_proibidos = "Painel do kit, servidores de terceiros, contas de cooperados, sistemas internos da cooperativa"  #@param {type:"string"}
tecnicas_proibidas = "Login com credenciais alheias, download de dados de vítimas, contato com suspeitos, perfis de fachada"  #@param {type:"string"}
pessoas_proibidas = "Qualquer pessoa física, incluindo o suposto operador do kit e seus familiares"  #@param {type:"string"}
gatilho_1 = "credenciais ou dados de cooperados expostos"  #@param {type:"string"}
escalar_1 = "Marta Quintão e jurídico da cooperativa; polícia via cooperativa"  #@param {type:"string"}
gatilho_2 = "qualquer material envolvendo menor de idade"  #@param {type:"string"}
escalar_2 = "SaferNet Brasil e Disque 100; paro a análise imediatamente"  #@param {type:"string"}
onde_guarda = "Pasta cifrada do caso na Vigia"  #@param {type:"string"}
quem_ve = "Eu, Rafa (sênior) e a gerência de risco da cooperativa"  #@param {type:"string"}
quando_apaga = "90 dias após o encerramento do caso, salvo pedido de preservação"  #@param {type:"string"}
base_legal_lgpd = "Art. 7º, IX (legítimo interesse da cooperativa na prevenção à fraude)"  #@param {type:"string"}

import hashlib
from datetime import datetime, timezone
from pathlib import Path

agora_utc = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Se a carta existir aqui no Colab (você subiu pelo ícone de pasta), calculamos o hash dela.
carta = Path(evidencia_escrita)
hash_carta = (hashlib.sha256(carta.read_bytes()).hexdigest()
              if carta.is_file() else "(suba o arquivo para calcular)")

registro = f"""# Registro de autorização

Contrato / ID do caso: {caso_id}
Autorizado por (nome, cargo, organização): {autorizado_por}
Autoridade dessa pessoa para conceder acesso: {autoridade}
Data da autorização (UTC): {agora_utc}
Evidência escrita da autorização (nome do arquivo + SHA-256): {evidencia_escrita} — {hash_carta}

## Dentro do escopo
- Sistemas / dados que posso examinar: {sistemas_permitidos}
- Técnicas que posso usar: {tecnicas_permitidas}

## Fora do escopo
- Sistemas / dados que não posso tocar: {sistemas_proibidos}
- Técnicas que não posso usar: {tecnicas_proibidas}
- Pessoas que não posso pesquisar: {pessoas_proibidas}

## Condições de parada
- Se eu encontrar {gatilho_1}, eu paro e escalo para {escalar_1}.
- Se eu encontrar {gatilho_2}, eu paro e escalo para {escalar_2}.

## Tratamento dos dados
- Onde a evidência fica guardada: {onde_guarda}
- Quem pode vê-la: {quem_ve}
- Quando é apagada: {quando_apaga}
- Base legal LGPD do tratamento: {base_legal_lgpd}
"""

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/registro-de-autorizacao.md")
saida.write_text(registro, encoding="utf-8")

# O hash do próprio registro: se alguém mexer depois, dá para provar.
hash_registro = hashlib.sha256(saida.read_bytes()).hexdigest()
print(registro)
print(f"💾 Salvo em {saida}")
print(f"🔏 SHA-256 do registro: {hash_registro}")

## 4. ✍️ Minha linha

A pergunta do Dia 1: ***o que me faria parar, mesmo que tecnicamente eu pudesse continuar?***

Escreva 3 a 5 frases. Você vai reler isto no checkpoint da Fase 5.

In [ ]:
#@title Minha linha { display-mode: "form" }
minha_linha = "Eu paro quando percebo que estou pesquisando uma pessoa e não um padrão. Eu paro diante de qualquer credencial alheia, mesmo exposta. Eu paro quando a justificativa começa com 'é para ajudar'."  #@param {type:"string"}

from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/minha-linha.md").write_text("# Minha linha\n\n" + minha_linha + "\n", encoding="utf-8")
print("💾 Salvo em notas/minha-linha.md")

# No Colab, baixa os dois arquivos para você subir no seu fork.
try:
    from google.colab import files
    files.download("notas/registro-de-autorizacao.md")
    files.download("notas/minha-linha.md")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta notas/)")

## ✅ Antes de fechar o Dia 1

- [ ] Quiz com pelo menos 12/14
- [ ] `notas/registro-de-autorizacao.md` gerado e enviado ao seu fork
- [ ] `notas/minha-linha.md` escrito com as suas palavras (não deixe o texto de exemplo!)
- [ ] Cartão do Dia 01 em **Artefato construído** no GitHub Projects
- [ ] `status` do dia 1 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 2):** disciplina de evidência, da coleta ao registro de custódia. Rafa vai pedir o hash de tudo.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>